# Can a seemingly random cortex self-organise?

Salt-and-pepper cortex looks like someone shook the orientation map in a bag 🧂. Our question is whether that apparent randomness could nevertheless be the product of an orderly developmental process.

**The short answer is yes.** This demo follows a model in which self-organisation builds a fabric of tiny, interconnected cortical domains. The structure is easy to see on an ideal lattice, but modest neuronal displacement can hide it from spatial measurements without destroying the representation itself.

Each section tells that story first. Open **Technical details** only when you want the machinery underneath.

<details>
<summary><strong>Technical details</strong></summary>

This notebook is a complete training-and-presentation workflow for micro-GCAL. If the completed archive is absent, it trains a 100×100 V1 sheet for two epochs; otherwise it loads the archived run. Code cells contain configuration and public helper calls only. Collection, checkpointing, measurements, plotting, animation, synthetic probes, and displacement analyses live in `helpers/microdomain_demo.py`.

</details>


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'demo_microdomains' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo_microdomains.helpers.microdomain_demo import (
    animation_html,
    animate_dimensionality_learning,
    animate_map_learning,
    animate_rotating_umap_grid,
    animate_scattered_map_learning,
    animate_synthetic_learning,
    animate_weight_learning,
    animate_robustness_learning,
    ensure_github_assets,
    estimate_demo_archive_gib,
    github_demo_config,
    macaque_displacement_table,
    plot_lgn_inputs_and_statistics,
    plot_macaque_displacement_links,
    plot_macaque_displacement_summary,
    run_macaque_displacement_demo,
    train_or_load_demo,
)

## 0. Build the model—or load one we prepared earlier

We built the model to test the idea directly. Running this notebook can reproduce both phases: first self-organisation, then the measurements and animations below. If the finished run is already present, we skip the several-hour training step and get straight to the cortex.

<details>
<summary><strong>Technical details</strong></summary>

The canonical configuration uses a 100×100 V1 microdomain sheet, 80×80 natural-image crops, two independently shuffled epochs, and 100 learning snapshots. `train_or_load_demo` treats a completed archive as immutable: it loads valid existing data and trains only when that archive is absent or incomplete.

</details>


In [ ]:
config = github_demo_config()
estimate_demo_archive_gib(config)

In [ ]:
archive = train_or_load_demo(config)
archive.manifest['status'], archive.n_frames

In [ ]:
assets = ensure_github_assets(archive)
assets

## 1. Start with the visual input

Development begins with sparse patches of natural scenes. A simple LGN-like stage highlights local contrast and normalises it, so the cortex receives edges and texture rather than raw photographs. These are the ingredients from which the map must organise itself—no orientation labels are supplied.

<details>
<summary><strong>Technical details</strong></summary>

Natural-image crops are passed through an ON-centre Laplacian-of-Gaussian filter and divisive gain control. The fixed held-out examples below provide a common reference for intensity, exact-zero sparsity, spatial power, and the number of principal components needed to explain 95% of LGN variance.

</details>

<img src="demo_assets/microdomain/lgn_inputs.png" alt="LGN inputs and statistics" width="800">


In [ ]:
plot_lgn_inputs_and_statistics(archive)

## 2. Watch tiny domains organise themselves

Repeated input and recurrent interaction gradually turn the sheet into a fabric of small orientation domains. The pattern is fine, but it is not random: the Fourier ring reveals a preferred spatial scale, while the retinotopic fishnet is locally bent yet globally stays in order.

<details>
<summary><strong>Technical details</strong></summary>

For each input, activity settles through short-range excitation, plastic longer-range inhibition, and selective cross-domain excitation before Hebbian and homeostatic updates. From left to right the animation shows orientation preference, its spatial Fourier power, horizontal receptive-field position, and the central quarter of the retinotopic fishnet. In the Fourier panel, black means power and white means none.

</details>

![Orientation map formation](demo_assets/microdomain/map_learning.gif)


In [ ]:
animation_html(animate_map_learning(archive))

## 3. See what the connections learn

The map is not painted onto the cortex from outside. Afferent receptive fields become selective as the sheet learns what to extract from the LGN, while cross-domain excitation learns which separated patches should cooperate. Tiny domains become an interconnected fabric rather than isolated dots.

<details>
<summary><strong>Technical details</strong></summary>

The left canvas follows sampled LGN→V1 afferent weights after independent zero-centred normalisation. The right canvas follows source-centred cross-domain excitatory fields. Only complete interior crops are used, with no border padding. Both use the same perceptually uniform scale with exact zero shown in white.

</details>

![Afferent and cross-domain plasticity](demo_assets/microdomain/weight_learning.gif)


In [ ]:
animation_html(animate_weight_learning(archive))

## 4. Ask whether the code is useful

An organised map should retain information, not merely look pretty. We therefore present the same synthetic face throughout learning and try to reconstruct it from settled cortical activity. The face is just the visual mascot; the curve reports average fidelity across the full fixed evaluation set. As learning proceeds, the population code becomes easier to decode. It learns to smile back 🙂.

<details>
<summary><strong>Technical details</strong></summary>

At every snapshot a concurrently trained nonlinear decoder maps settled V1 activity back to LGN space. The first three panels show the fixed face, its cortical response, and its reconstruction. The last panel is the mean input–reconstruction cosine similarity across all held-out reconstruction examples—not the face score alone. No intermediate image is interpolated.

</details>

![Tracked reconstruction fidelity](demo_assets/microdomain/synthetic_learning.gif)


In [ ]:
animation_html(animate_synthetic_learning(archive))

## 5. Count the population's degrees of freedom

The learned activity has regular correlational structure: many neurons vary together rather than independently. PCA makes that structure visible. Because fewer effective dimensions are needed to describe V1 activity than the LGN input, the recurrent sheet has compressed the representation without simply erasing it.

<details>
<summary><strong>Technical details</strong></summary>

The first three panels show the spatial loadings of the leading principal components through learning. The final plot tracks the number of components required to explain 95% of V1 variance and compares it with the fixed LGN value. Their ratio is a unitless estimate of relative representational dimensionality—roughly, the fraction of input degrees of freedom retained by the cortical code.

</details>

![PCA geometry and dimensionality](demo_assets/microdomain/dimensionality.gif)


In [ ]:
animation_html(animate_dimensionality_learning(archive))

## 6. Give the recurrent dynamics a noisy day

Efficient representations are useful only if they are reliable. Selective excitation helps the network settle towards a similar population response even when noise is injected during the recurrent interaction. The emerging code is therefore not only compact, but increasingly stable.

<details>
<summary><strong>Technical details</strong></summary>

The same fixed LGN input and the same noise realisation are used at every learning snapshot. The middle panels compare matched 20×20 windows of clean activity and activity with noise intensity 0.06. The final plot tracks the population cosine similarity between clean and noisy settled codes through training.

</details>

![Noise robustness](demo_assets/microdomain/robustness.gif)


In [ ]:
animation_html(animate_robustness_learning(archive))

## 7. So why does real cortex look salt-and-pepper?

So far the tiny domains are suspiciously tidy. Real neurons, however, do not remain on a perfect square lattice: they disperse as cortex develops. Moving cells by an average of only two model positions preserves short-range clustering—compatible with cellular-scale observations such as Ringach et al. (2016)—but wipes out the global periodic signature. Same learned map, much messier seating plan.

<details>
<summary><strong>Technical details</strong></summary>

The seeded Gaussian local permutation used in `wiring_efficiency.ipynb` is calibrated to the requested mean displacement in lattice units and held fixed throughout the animation. It is applied consistently to orientation preference and receptive-field identity. The panels show the displaced orientation map, its Fourier power, horizontal retinotopy, and the zoomed fishnet. Set `scatter_displacement` below to change the mean displacement.

</details>

![Scatter permutation](demo_assets/microdomain/scattered_learning.gif)


In [ ]:
scatter_displacement = 2.0
animation_html(
    animate_scattered_map_learning(
        archive, mean_displacement=scatter_displacement
    )
)

## 8. How much displacement might there be in cortex?

We do not know the developmental scatter directly, but dense macaque V1 recordings let us make a simple estimate. We compare each measured neuron's location with the nearest place where a smooth underlying orientation map predicts the same preference. The resulting distances provide a data-driven scale for positional displacement—not a perfect ruler, but a useful reality check.

<details>
<summary><strong>Technical details</strong></summary>

The analysis uses the three densest 850×850 µm fields (`MB_1`, `MB_2`, and `MA_2`) from the public Chen et al. macaque V1 cellular-orientation dataset. A leave-one-cell-out circular map is smoothed with a shared Gaussian σ of 100 µm. Every significantly tuned cell is shown, but contour matches above 350 µm are excluded from the reported mean and the linked subsample so rare extreme matches do not dominate the estimate.

| Dataset | Included cells | Mean displacement |
|---|---:|---:|
| `MB_1` | 792 | 91.4 µm |
| `MB_2` | 729 | 82.5 µm |
| `MA_2` | 395 | 92.3 µm |

</details>

<img src="demo_assets/microdomain/macaque_displacement_summary.png" alt="Macaque cellular scatter and smooth maps" width="1050">


In [ ]:
macaque_smoothing_sigma_um = 100.0
macaque_max_displacement_um = 350.0
macaque_results = run_macaque_displacement_demo(
    smoothing_sigma_um=macaque_smoothing_sigma_um,
    max_displacement_um=macaque_max_displacement_um,
)
plot_macaque_displacement_summary(macaque_results)
macaque_displacement_table(macaque_results)

### Turn the estimate into visible journeys

The summary above becomes easier to interpret when a few individual journeys are drawn. Each black line asks: “if this cell belonged to the smooth underlying map, where would its preferred orientation place it?”

<details>
<summary><strong>Technical details</strong></summary>

A reproducible sparse subset of included cells is overlaid on a faint smooth map. Each solid black segment starts at the measured soma and ends at the nearest exact same-orientation location predicted by the leave-one-cell-out map; a black × marks that endpoint. Only 20 examples per field are drawn for legibility, while the titles retain the included-cell means.

</details>

<img src="demo_assets/microdomain/macaque_displacement_links.png" alt="Sparse macaque displacement links" width="1050">


In [ ]:
macaque_link_examples = 20
plot_macaque_displacement_links(
    macaque_results, n_links=macaque_link_examples, map_alpha=0.28
)

## 9. Hidden on the sheet does not mean absent from the code

After displacement, the spatial map looks salt-and-pepper—but the responses still live on an orderly manifold. Static gratings, the topological simulation, the displaced microdomain simulation, and real mouse V1 all produce similarly smooth, folded response geometries. The lattice pattern may vanish, yet the representation remains regular in feature space.

<details>
<summary><strong>Technical details</strong></summary>

The four panels show separately fitted 3D UMAPs of raw full-phase grating pixels, topographic-model responses, salt-and-pepper-model responses, and high-arousal responses from Stringer et al. (2021) random-phase recording 1. Colour denotes orientation modulo 180°. The stimulus and model embeddings are compatible with the twisted angle–phase identification of a Klein bottle; the mouse data lack matched phase labels, so their resemblance is qualitative rather than a formal topological test. Synchronized rotation exposes the full geometry without distorting the fitted axis scales.

</details>

<img src="demo_assets/microdomain/rotating_umap.gif" alt="Rotating four-panel 3D UMAP comparison" width="1050">


In [ ]:
umap_rotation_frames = 48
umap_animation_asset = assets['rotating_umap']

## Conclusion: order can hide in plain sight

The demo began with an apparent contradiction: how could salt-and-pepper cortex emerge through self-organisation if self-organisation is supposed to make smooth maps? The model suggests a simple answer. Development can build selective receptive fields, efficient population codes, robust recurrent dynamics, and structured cross-domain coupling at a very fine spatial scale. Modest neuronal displacement then hides the periodic layout without undoing what the network learned.

So salt-and-pepper need not mean **structureless**. It may mean **structured at a scale that cortical position no longer reveals**.

<details>
<summary><strong>Technical takeaway</strong></summary>

Micro-GCAL extends classical recurrent self-organisation with learned selective cross-domain excitation. On the ideal sheet it produces small coupled domains, smooth retinotopy, reduced effective dimensionality, improving reconstruction fidelity, and robust settled codes. Local positional disorder masks the high-frequency spatial signatures, while local tuning similarity and smooth population-response geometry remain measurable. This is a model-based hypothesis—not proof that every biological salt-and-pepper map develops this way—but it gives the hypothesis concrete, testable signatures.

</details>
